# Jinja Fish-Export Cooperative: Sustainable Harvest & Revenue Risk Model

Replaces the old Fibonacci-stock / raw-variance-threshold approach with:

1. A discrete logistic growth-with-harvesting stock model
2. A bounded random-walk price model
3. Dispersion statistics (mean, median, variance, sd, CV)
4. A CV-based risk classifier and Monte Carlo Value-at-Risk
5. Scenario analysis across harvest rates, benchmarked against MSY
6. A closed-season extension

All reusable logic lives in `src/`; this notebook only orchestrates and presents results.

In [8]:
import matplotlib.pyplot as plt
import pandas as pd

from src.stock import fibonacci, msy_benchmarks
from src.price import PriceModel, weekly_revenue
from src.risk import revenue_stats, RiskAssessor
from src.simulation import run_scenarios, monte_carlo_annual_revenue, closed_season_comparison
from src.plotting import plot_stock_trajectories, plot_revenue_histogram

%matplotlib inline

## 1. Old baseline: Fibonacci "stock"
Grows without bound and has no link to biology or harvesting, which is why it was replaced.

In [ ]:
fibonacci(15)

## 2. Scenario analysis across harvest rates

In [ ]:
harvest_rates = [0.05, 0.10, 0.20, 0.30]
results, msy, h_at_msy = run_scenarios(harvest_rates)
print(f"Theoretical MSY = rK/4 = {msy:.0f} tonnes/week, reached at steady state when h = r/2 = {h_at_msy:.2f}")

summary = pd.DataFrame({
    h: {
        "Final stock (t)": r["final_stock"],
        "Steady harvest / wk (t)": r["steady_state_harvest_wk"],
        "Mean annual revenue (UGX)": r["mean_annual_revenue"],
        "CV": r["cv"],
        "Risk": r["risk_class"],
        "5% VaR (UGX)": r["var_5pct"],
    } for h, r in results.items()
}).T
summary.index.name = "h"
summary.style.format({
    "Final stock (t)": "{:,.1f}", "Steady harvest / wk (t)": "{:,.1f}",
    "Mean annual revenue (UGX)": "{:,.0f}", "CV": "{:.3f}", "5% VaR (UGX)": "{:,.0f}",
})

## 3. Stock trajectories

In [ ]:
fig = plot_stock_trajectories(results, path="figures/stock_trajectories.png")
plt.show()

## 4. Revenue distribution and 5% VaR (h = 0.20)

In [ ]:
fig = plot_revenue_histogram(h=0.20, path="figures/revenue_histogram.png")
plt.show()

## 5. Dispersion statistics on a single realised price path
One seeded path of weekly revenue at h = 0.20, using the `PriceModel` class.

In [ ]:
from src.stock import FishStock

stock = FishStock()
harvests = stock.simulate(52, h=0.20)
prices = PriceModel(seed=42).simulate(52)
stats_one_path = revenue_stats(weekly_revenue(harvests, prices))
stats_one_path["risk"] = RiskAssessor.classify(stats_one_path["cv"])
stats_one_path

## 6. Extension: 8-week closed season over 5 years (h = 0.20)

In [ ]:
cs = closed_season_comparison(h=0.20, years=5, closed_weeks=8)
print(f"No closed season   -> final stock: {cs['final_stock_open']:.0f} t | mean 5yr revenue: {cs['mean_revenue_open']:,.0f} UGX")
print(f"8-wk closed season -> final stock: {cs['final_stock_closed']:.0f} t | mean 5yr revenue: {cs['mean_revenue_closed']:,.0f} UGX")
print(f"Revenue change: {cs['revenue_change_pct']:+.2f}%   Final-stock change: {cs['stock_change_pct']:+.2f}%")